# NL2SQL Training (Kaggle T4)

All real code lives in the GitHub repo (`src/`). This notebook just orchestrates.

## Before running — one-time setup (right sidebar)
1. **Accelerator** → GPU T4 x2
2. **Internet** → On
3. **Secrets** → Add these 3:
   - `HF_TOKEN` — HuggingFace write token (huggingface.co/settings/tokens)
   - `WANDB_API_KEY` — W&B API key (wandb.ai/settings)
   - `HF_HUB_REPO` — `Akashkar00/nl2sql-qwen-coder-7b-qlora`
4. **Add Input** → add your Kaggle dataset named `nl2sql-processed-data`
   (upload train.jsonl, eval_holdout.jsonl, spider_dev.jsonl from data/processed/ on your Mac)

In [ ]:
# Cell 1: secrets + clone repo
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['HF_HUB_REPO'] = secrets.get_secret('HF_HUB_REPO')

GIT_URL = 'https://github.com/Akashkar00/nl2sql-agent.git'

!cd /kaggle/working && git clone $GIT_URL || (cd /kaggle/working/nl2sql-agent && git pull)
%cd /kaggle/working/nl2sql-agent

In [ ]:
# Cell 2: install deps. Unsloth requires special install order on Kaggle.
!pip install -q --no-deps 'unsloth @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git'
!pip install -q -r requirements.txt
# Note: torch is pre-installed on Kaggle. Don't reinstall — wastes 5min.

In [ ]:
# Cell 3: copy pre-processed data from Kaggle dataset input
import shutil, os

os.makedirs("data/processed", exist_ok=True)
for f in ["train.jsonl", "eval_holdout.jsonl", "spider_dev.jsonl"]:
    shutil.copy(f"/kaggle/input/nl2sql-processed-data/{f}", f"data/processed/{f}")

!wc -l data/processed/*.jsonl
# Expected: train=8159, eval_holdout=500, spider_dev=1034

In [ ]:
# Cell 5: login to HF + W&B
from huggingface_hub import login as hf_login
import wandb
hf_login(token=os.environ['HF_TOKEN'])
wandb.login(key=os.environ['WANDB_API_KEY'])

In [ ]:
# Cell 6: TRAIN. ~6-8 hours on T4 for 2 epochs / 16K examples.
# Adapter pushed to HF Hub every save_steps — survives Kaggle session death.
!python -m src.train --config configs/training_config.yaml

In [ ]:
# Cell 7: quick eval on holdout (sanity check before full benchmark)
!python -m src.eval.run_benchmark \
    --eval_path data/processed/eval_holdout.jsonl \
    --model_name Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path $HF_HUB_REPO \
    --run_name finetuned_with_agent \
    --limit 200